In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler

In [ ]:
# Load your CSV
df = pd.read_csv("basic_shape_features.csv")

# Separate metadata and features
non_features = df[['filename', 'country']]
features = df.drop(columns= non_features)

# Standardize features
scaler = StandardScaler()
features_scaled = scaler.fit_transform(features)

features_scaled = pd.DataFrame(features_scaled, columns=features.columns)

In [ ]:
#detecting outliers in the shape space using Mahalanobis distance
import numpy as np
from scipy.spatial.distance import mahalanobis
from numpy.linalg import inv

X = features_scaled.values
cov_matrix = np.cov(X, rowvar=False)
inv_cov = inv(cov_matrix)
mean_distr = X.mean(axis=0)

distances = []

for i in range(len(X)):
    dist = mahalanobis(X[i], mean_distr, inv_cov)
    distances.append(dist)

df['mahal_dist'] = distances

# Chi-square threshold
from scipy.stats import chi2
threshold = chi2.ppf((1 - 0.001), df=X.shape[1])

df_clean = df[df['mahal_dist'] < np.sqrt(threshold)]

In [ ]:
# After QC 
non_features = df_clean[['filename', 'country']]
features_clean = df_clean.drop(columns= non_features)

# Standardize features
scaler = StandardScaler()
features_scaled_clean = scaler.fit_transform(features_clean)

features_scaled_clean = pd.DataFrame(features_scaled_clean, columns=features_clean.columns)

In [ ]:
#create distance matrix
from scipy.spatial.distance import pdist, squareform
from skbio import DistanceMatrix
ids = df_clean.index.astype(str)
distance_matrix = squareform(pdist(features_scaled_clean, metric='euclidean'))
dm = DistanceMatrix(distance_matrix, ids=ids)

grouping = non_features['country']
grouping.index = ids

In [ ]:
from skbio.stats.distance import permanova, permdisp

permanova_results = permanova(dm, grouping=grouping, permutations=999)
print(permanova_results)

dispersion_results = permdisp(dm, grouping=grouping, permutations=999)
print(dispersion_results)

In [ ]:
#pairwise permanova
import itertools
import pandas as pd
from skbio.stats.distance import permanova

results = []

countries = df_clean['country'].unique()

for c1, c2 in itertools.combinations(countries, 2):
    
    # Subset data
    mask = df_clean['country'].isin([c1, c2])
    subset_ids = df_clean[mask].index.astype(str)
    
    dm_subset = dm.filter(subset_ids)
    
    grouping_subset = df_clean.loc[mask, 'country']
    grouping_subset.index = subset_ids
    
    res = permanova(dm_subset, grouping_subset, permutations=999)
    
    results.append({
        'Group1': c1,
        'Group2': c2,
        'pseudo-F': res['test statistic'],
        'p-value': res['p-value']
    })

pairwise_results = pd.DataFrame(results)
print(pairwise_results)

In [ ]:
#correct for multiple testing (FDR).
from statsmodels.stats.multitest import multipletests

pairwise_results['p_adj'] = multipletests(
    pairwise_results['p-value'],
    method='fdr_bh'
)[1].round(4)

pairwise_results.to_csv('pairwise_table.csv', index=True)